__Dataset__: [Link](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [1]:
import numpy as np
import pandas as pd
import re
import string
from utils.constants import chat_words
import os

import nltk
NLTK_DIR = os.path.abspath("./env/nltk_data")
# Tokenizer
nltk.download('punkt', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
nltk.download('punkt_tab', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# Correct spelling
nltk.download('words', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# stopword removal
nltk.download('stopwords', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# download wordnet
nltk.download('wordnet', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)

nltk.data.path.insert(0, NLTK_DIR)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.corpus import words, stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from collections import Counter

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
DATAPATH = 'data/spam.csv'

dataset_v1 = pd.read_csv(
    DATAPATH,
    header=0,
    usecols=[0, 1],
    names=["label", "message"],
    encoding="utf-8",
    encoding_errors="ignore"
)

df = dataset_v1.copy()
df = df.dropna(subset=["label", "message"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str)

print(f"Messages: {len(df):,}")
print("Label counts:", df["label"].value_counts().to_dict())
df.sample(5)


Messages: 5,572
Label counts: {'ham': 4825, 'spam': 747}


,label,message
3986,spam,Ringtone Club: Gr8 new polys direct to your mo...
3394,ham,Then i buy.
4930,ham,"Good morning, my boytoy! How's those yummy lip..."
849,spam,Today's Offer! Claim ur 150 worth of discount ...
2164,ham,"Nothing really, just making sure everybody's u..."


In [3]:
def remove_number_middle_of_text(text):
    pattern = re.compile(r'[a-zA-Z]+')
    text = pattern.sub(r"", text)
    return " ".join(text)

def remove_htmltags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

def remove_urls(text):
    pattern = re.compile(r"https?:\/\/\S+|www\.\S+")
    return pattern.sub(r"", text)

def remove_punctuations(text):
    excluded_punctuations = string.punctuation
    return text.translate(str.maketrans(" ", " ", excluded_punctuations))

def eloborate_chat_conversion(texts):
    new_text = []
    token_words = texts.split()
    
    for word in token_words:
        key = word.upper()
        if key in chat_words:
            new_text.append(chat_words[key])
        else:
            new_text.append(word)
    return " ".join(new_text)


def corrected_spelling(texts):
    results = []
    correct_words = [w.lower() for w in words.words()]
    correct_set = set(correct_words)
    text_tokens = texts.split()
    
    for word in text_tokens:
        lower_word = word.lower()
        # keep the correctly spelled words
        if lower_word in correct_set or not lower_word.isalpha() or len(lower_word) < 2:
            results.append(lower_word)
            continue
        
        incorrect_bigrams = set(ngrams(lower_word, 2))
        best_score, best_match = 1.0, word
        
        for w in correct_words:
            if w[0] == lower_word[0]:
                correct_bigrams = set(ngrams(w, 2))
                score = jaccard_distance(incorrect_bigrams, correct_bigrams)
                if score < best_score:
                    best_score, best_match = score, w
        results.append(best_match)
    return " ".join(results)

def remove_stopwords(texts):
    cleaned = []
    text_tokens = texts.split()
    stop_words = set(stopwords.words('english'))
    for word in text_tokens:
        # word is already lowered
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)


def remove_emoji(texts):
    text_tokens = texts.split()
    text = " ".join(text_tokens)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    texts = emoji_pattern.sub(r'', text)
    return texts


def stemming_words(text_tokens):
    ps = PorterStemmer()
    stemmed_words = []
    return [stemmed_words.append(ps.stem(word)) for word in text_tokens]

def lemmatize_words(text_tokens):
    lemmatized_words = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for word in text_tokens:
        lemmatized_words.append(wordnet_lemmatizer.lemmatize(word, pos='v'))
    return lemmatized_words


def text_preprocessing(text):
    # lowercasing
    text = text.lower()
        
    # remove html tags
    text = remove_htmltags(text)
    
    # remove urls from text
    text = remove_urls(text)
    
    text = remove_punctuations(text)
    
    text = eloborate_chat_conversion(text)
    
    text = corrected_spelling(text)
    
    text = remove_stopwords(text)
    
    text = remove_emoji(text)
    
    tokenized_text = word_tokenize(text)
    
    # tokenized_text = stemming_words(tokenized_text)
    
    tokenized_text = lemmatize_words(tokenized_text)
    
    
    return tokenized_text


In [ ]:
# text_preprocessing("""helo amzing ADIH <p>hello world</p> <br/> https://helo_texts.com
# https://www.helo_texts.com
# http://localhost:5000"""+df['message'][0])
df['tokens'] = df['message'].apply(text_preprocessing)
# corpus_tokens = (df['message'].apply(text_preprocessing)).tolist()

In [11]:
corpus_tokens = df['tokens'].tolist()
df['cleaned_text'] = df['tokens'].apply(' '.join)
# df['cleaned_text'].iloc[4138]

In [ ]:
# all_words = [token for tokens in corpus_tokens for token in tokens]

In [6]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])
df['encoded_label'] = y

#### OneHotEncoder

In [9]:
all_tokens = [[tok] for tokens in corpus_tokens for tok in tokens if len(tok) > 0]   # shape (n_tokens, 1)

encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype=np.int8)
encoder.fit(all_tokens)

vocab_size = len(encoder.categories_[0])

def document_to_onehot(tokens, ohe=encoder):
    if not tokens:
        return np.zeros(len(ohe.categories_[0]), dtype=np.int8)
    rows = ohe.transform([[t] for t in tokens])
    return (rows.sum(axis=0).A1 > 0).astype(np.int8)

# ohe_mat = document_to_onehot(corpus_tokens, encoder)

# df['ohe'] = df['tokens'].apply(document_to_onehot)
X_ohe = np.vstack(df['tokens'].apply(document_to_onehot))
print(f"After apply the OneHotEncoder: {X_ohe.shape}")


After apply the OneHotEncoder: (5572, 7005)


#### Bag Of Words

In [14]:
corpus = df['cleaned_text']

cv = CountVectorizer()
cv_ngram = CountVectorizer(ngram_range=(1, 2))
X_bow = cv.fit_transform(corpus)
X_bow_ngram = cv_ngram.fit_transform(corpus)

In [57]:
X_bow = X_bow.toarray()

In [ ]:
print(f'After apply the BoW(Bag of Word) the shape: {X_bow.shape}')
print(f'Vocabulary size: {len(cv.vocabulary_)}')
# cv.get_feature_names_out()[1000]

After apply the BoW(Bag of Word) the shape: (5572, 6972)
Vocabulary size: 6972


### TF-IDF

In [43]:
tfidf = TfidfVectorizer()

tfidf_transformed = tfidf.fit_transform(df['cleaned_text'])

In [44]:
X_tfidf = tfidf_transformed.toarray()

In [50]:
len(tfidf.vocabulary_)

6972

In [55]:
new_text = ["Hello World how are you!"]

tfidf.transform(new_text).toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 6972))